<div style='text-align: center; padding: 30px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); border-radius: 15px; margin: 10px 0; box-shadow: 0 10px 30px rgba(0,0,0,0.2);'>
  <h1 style='color: white; margin: 0 0 8px 0; font-size: 2.5em;'>🎤 MOSS-TTS v1.5 - Standalone Foundation Model</h1>
  <h3 style='color: #f0f0f0; margin: 0 0 5px 0; font-weight: 400;'>Kaggle T4 x2 GPU Edition - Created by <strong>AIQUEST Academy</strong></h3>
  <p style='color: #ddd; margin: 0; text-align: center;'>High-Fidelity 48 kHz Stereo TTS and Zero-Shot Voice Cloning</p>
</div>

<div align="center">
  <img src="https://img.shields.io/badge/AIQUESTAcademy-blueviolet?style=for-the-badge&logo=youtube&logoColor=white" />
  <img src="https://img.shields.io/badge/Kaggle-T4%20GPU%20x2-20BEFF?style=for-the-badge&logo=kaggle&logoColor=white" />
  <br>
  <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1">
    <img src="https://img.shields.io/badge/Subscribe%20on%20YouTube-FF0000?style=for-the-badge&logo=youtube&logoColor=white" />
  </a>
  &nbsp;
  <a href="https://x.com/aiquestacademy">
    <img src="https://img.shields.io/badge/Follow%20on%20X-000000?style=for-the-badge&logo=x&logoColor=white" />
  </a>
</div>

### Setup Instructions
1. **Settings -> Accelerator -> GPU T4 x2** (MUST select dual T4 GPUs)
2. Run all cells **top to bottom**
3. Use the public Gradio link to open the web interface

---

## ⚙️ Cell 1 - Environment and GPU Memory Optimization

In [ ]:
# Cell 1: Check GPU and Optimize Environment Memory
import os
import gc
import sys
import torch
import psutil

print("=== Kaggle T4 Environment Setup ===")
print(f"Python Version: {sys.version}")
print(f"PyTorch Version: {torch.__version__}")
print(f"RAM: {psutil.virtual_memory().total / 1024**3:.1f} GB total, {psutil.virtual_memory().available / 1024**3:.1f} GB available")

# Optimize virtual memory allocations and drop page caches
os.system("echo 3 | sudo tee /proc/sys/vm/drop_caches > /dev/null 2>&1")
os.system("echo 1 | sudo tee /proc/sys/vm/overcommit_memory > /dev/null 2>&1")
gc.collect()

# Prevent memory fragmentation and aggressive VRAM limits
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,garbage_collection_threshold:0.6"
os.environ["MALLOC_TRIM_THRESHOLD_"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

if torch.cuda.is_available():
    device_count = torch.cuda.device_count()
    print(f"Number of GPUs Available: {device_count}")
    for i in range(device_count):
        device_name = torch.cuda.get_device_name(i)
        total_memory = torch.cuda.get_device_properties(i).total_memory / 1e9
        print(f"  GPU {i}: {device_name} ({total_memory:.2f} GB VRAM)")
    if device_count < 2:
        print("WARNING: Only 1 GPU detected. Go to Settings -> Accelerator -> GPU T4 x2 to enable dual GPUs.")
else:
    print("WARNING: No GPU detected. Go to Settings -> Accelerator -> GPU T4 x2")

# Force efficient attention options for Turing GPUs (T4)
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)

print("Environment setup and memory footprint optimizations applied!")

## 📦 Cell 2 - Install Dependencies

In [ ]:
# Cell 2: Install required packages
print("Installing system packages (libfst-dev, ffmpeg)...")
!apt-get update && apt-get install -y libfst-dev ffmpeg libsndfile1-dev > /dev/null 2>&1

print("Installing pynini and WeTextProcessing (required for text normalization)...")
!pip install -q pynini WeTextProcessing

print("Installing transformers, accelerate, soundfile, torchaudio, hf-transfer, pydantic, gradio, and bitsandbytes...")
!pip install -q transformers accelerate soundfile torchaudio hf-transfer pydantic==2.10.6 gradio==5.7.1 bitsandbytes

print("Environment setup completed successfully!")

## 📥 Cell 3 - Load MOSS-TTS Model (Dual GPU Balanced)

In [ ]:
# Cell 3: Load models with cache redirection
import os
import torch
from transformers import AutoModel, AutoProcessor, BitsAndBytesConfig

# Redirect model cache to large scratch space to avoid filling up the 20GB root overlay
os.environ["HF_HOME"] = "/kaggle/tmp/hf_cache"

# Enable high-speed Hugging Face download (hf-transfer) and mirror endpoint
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

MODEL_ID = "OpenMOSS-Team/MOSS-TTS-v1.5"

# Device plan: Tokenizer on CPU (to save VRAM), Model sharded across both GPUs
device_tok = "cpu"
device_model = "cuda:0" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

print(f"Loading processor and model (Sharded FP16 over GPU T4 x2, Tokenizer on CPU)...")

# Load processor
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)

# Keep the processor's audio tokenizer on CPU in float32 precision to save VRAM and prevent bias type mismatch
if hasattr(processor, "audio_tokenizer") and processor.audio_tokenizer is not None:
    processor.audio_tokenizer = processor.audio_tokenizer.to(device_tok).float()
    print(f"Processor audio tokenizer successfully moved to CPU in float32")

# Load main autoregressive model sharded automatically via device_map="auto"
model = AutoModel.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    torch_dtype=dtype,
    low_cpu_mem_usage=True,
    device_map="auto" if torch.cuda.is_available() else "cpu",
    attn_implementation="sdpa"
).eval()

# Apply Multi-GPU device safety monkeypatches to handle accelerate sharding
import types

def fixed_get_input_embeddings(self, input_ids: torch.LongTensor) -> torch.Tensor:
    base_embed = self.language_model.get_input_embeddings()
    inputs_embeds = base_embed(input_ids[..., 0].to(next(base_embed.parameters()).device)).to(dtype=dtype)
    for i, embed_layer in enumerate(self.emb_ext):
        vq_device = next(embed_layer.parameters()).device
        idx = input_ids[..., i + 1].to(vq_device)
        val = embed_layer(idx).to(device=inputs_embeds.device, dtype=inputs_embeds.dtype)
        inputs_embeds = inputs_embeds + val
    return inputs_embeds

model.get_input_embeddings = types.MethodType(fixed_get_input_embeddings, model)

original_forward = model.forward

def fixed_forward(self, *args, **kwargs):
    input_ids = kwargs.get("input_ids", None)
    if input_ids is None and len(args) > 0:
        input_ids = args[0]
    outputs = original_forward(*args, **kwargs)
    if outputs.logits is not None:
        target_device = input_ids.device if input_ids is not None else next(self.parameters()).device
        outputs.logits = [logit.to(target_device) for logit in outputs.logits]
    return outputs

model.forward = types.MethodType(fixed_forward, model)

# Apply numerical stability monkeypatch for sample_token to prevent NaN/inf CUDA assertions
import sys
import torch.nn.functional as F

target_modules = []
for name, module in list(sys.modules.items()):
    if ("inference_utils" in name or "modeling_moss_tts" in name) and ("openmoss" in name.lower()):
        target_modules.append((name, module))

if target_modules:
    # Get helpers from the actual inference_utils module
    utils_module = None
    for name, module in target_modules:
        if "inference_utils" in name:
            utils_module = module
            break
    if utils_module is None:
        utils_module = target_modules[0][1]
        
    def fixed_sample_token(logits, prev_tokens=None, repetition_penalty=1.0, top_p=None, top_k=None, do_sample=True):
        if torch.isnan(logits).any():
            logits = torch.nan_to_num(logits, nan=0.0, posinf=1e4, neginf=-1e4)
            
        vocab_size = logits.size(-1)
        if prev_tokens is not None and repetition_penalty != 1.0:
            logits = utils_module.apply_repetition_penalty_delay_pattern(
                logits, prev_tokens, repetition_penalty
            )
            
        if not do_sample:
            return torch.argmax(logits, dim=-1)
            
        original_shape = logits.shape
        reshaped_logits = logits.view(-1, vocab_size)
        
        # Check for rows that are entirely -inf (softmax will output NaNs on them)
        all_neginf = (reshaped_logits == float('-inf')).all(dim=-1)
        if all_neginf.any():
            reshaped_logits[all_neginf, 0] = 0.0
            
        if top_k is not None and top_k > 0:
            reshaped_logits = utils_module.apply_top_k(reshaped_logits, top_k)
            
        if top_p is not None and top_p < 1.0:
            reshaped_logits = utils_module.apply_top_p_optimized(reshaped_logits, top_p)
            
        # Cast to float32 before softmax and multinomial for numerical stability in float16/bfloat16
        probs = F.softmax(reshaped_logits.float(), dim=-1)
        
        if torch.isnan(probs).any():
            probs = torch.nan_to_num(probs, nan=0.0)
            sums = probs.sum(dim=-1, keepdim=True)
            sums[sums == 0.0] = 1.0
            probs = probs / sums
            
        sums = probs.sum(dim=-1)
        zero_sums = sums == 0.0
        if zero_sums.any():
            probs[zero_sums, 0] = 1.0
            
        next_tokens = torch.multinomial(probs, num_samples=1)
        return next_tokens.view(original_shape[:-1])
        
    for name, module in target_modules:
        if hasattr(module, "sample_token"):
            module.sample_token = fixed_sample_token
            print(f"Successfully monkeypatched sample_token in module: {name}")
else:
    print("Warning: Could not find target modules to patch.")

print("Models loaded successfully, sharded, and device-safety patched!")

## 🎛️ Cell 4 - Launch Gradio Web UI

In [ ]:
# Cell 4: Launch Gradio App with Soft Theme and Fast Preset
import time
import gc
import tempfile
from datetime import datetime
import traceback
import gradio as gr
import torchaudio
from transformers import GenerationConfig

# Subclass GenerationConfig to match custom MOSS-TTS Delay configuration parameters
class DelayGenerationConfig(GenerationConfig):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.layers = kwargs.get("layers", [{} for _ in range(32)])
        self.do_samples = kwargs.get("do_samples", None)
        self.n_vq_for_inference = 32

# Supported languages list
LANGUAGES = [
    "Auto (omit)", "English", "Chinese", "Japanese", "Korean", "French", "German",
    "Spanish", "Italian", "Portuguese", "Russian", "Arabic", "Cantonese",
    "Vietnamese", "Thai", "Turkish", "Hindi", "Indonesian", "Malay",
    "Dutch", "Swedish", "Polish", "Danish", "Finnish", "Norwegian",
    "Czech", "Greek", "Hungarian", "Romanian", "Slovak", "Ukrainian",
    "Hebrew"
]

# (Presets removed - 32 RVQ used by default)

def to_device(obj, device):
    """Recursively map PyTorch tensors inside nested structures to target device"""
    if isinstance(obj, torch.Tensor):
        return obj.to(device)
    elif isinstance(obj, list):
        return [to_device(x, device) for x in obj]
    elif isinstance(obj, dict):
        return {k: to_device(v, device) for k, v in obj.items()}
    elif isinstance(obj, tuple):
        return tuple(to_device(x, device) for x in obj)
    return obj

ZH_TOKENS_PER_CHAR = 3.098411951313033
EN_TOKENS_PER_CHAR = 0.8673376262755219

def detect_text_language(text: str) -> str:
    import re
    zh_chars = len(re.findall(r"[\u4e00-\u9fff]", text))
    en_chars = len(re.findall(r"[A-Za-z]", text))
    if zh_chars == 0 and en_chars == 0:
        return "en"
    return "zh" if zh_chars >= en_chars else "en"

def supports_duration_control(mode_with_reference: str) -> bool:
    return mode_with_reference not in ["Continuation", "Continuation + Clone"]

def estimate_duration_tokens(text: str) -> tuple[str, int, int, int]:
    normalized = text or ""
    effective_len = max(len(normalized), 1)
    language = detect_text_language(normalized)
    factor = ZH_TOKENS_PER_CHAR if language == "zh" else EN_TOKENS_PER_CHAR
    default_tokens = max(1, int(effective_len * factor))
    min_tokens = max(1, int(default_tokens * 0.5))
    max_tokens = max(min_tokens, int(default_tokens * 1.5))
    return language, default_tokens, min_tokens, max_tokens

def update_duration_controls(
    enabled: bool,
    text: str,
    current_tokens: float | int | None,
    mode_with_reference: str,
):
    if not supports_duration_control(mode_with_reference):
        return (
            gr.update(visible=False),
            "Duration control is disabled for Continuation modes.",
            gr.update(value=False, interactive=False),
        )

    checkbox_update = gr.update(interactive=True)
    if not enabled:
        return gr.update(visible=False), "Duration control is disabled.", checkbox_update

    language, default_tokens, min_tokens, max_tokens = estimate_duration_tokens(text)
    if current_tokens is None or int(current_tokens) == 1:
        slider_value = default_tokens
    else:
        slider_value = int(current_tokens)
        slider_value = max(min_tokens, min(max_tokens, slider_value))

    language_label = "Chinese" if language == "zh" else "English"
    hint = (
        f"Duration control enabled | detected language: {language_label} | "
        f"default={default_tokens}, range=[{min_tokens}, {max_tokens}]"
    )
    return (
        gr.update(
            visible=True,
            minimum=min_tokens,
            maximum=max_tokens,
            value=slider_value,
            step=1,
        ),
        hint,
        checkbox_update,
    )

def render_mode_hint(reference_audio: str | None, mode_with_reference: str):
    if not reference_audio:
        return "Current mode: **Direct Generation** (no reference audio uploaded)"
    if mode_with_reference == "Clone":
        return "Current mode: **Clone** (speaker timbre will be cloned from the reference audio)"
    return f"Current mode: **{mode_with_reference}**  \n> Continuation mode is active. Make sure the reference audio transcript is prepended to the input text."

def generate_speech(
    text,
    language,
    reference_audio,
    mode_with_reference,
    speed=1.0,
    text_temp=1.2,
    text_top_p=1.0,
    text_top_k=50,
    audio_temp=1.7,
    audio_top_p=0.9,
    audio_top_k=25,
    audio_repetition_penalty=1.0,
    duration_control_enabled=False,
    expected_tokens_val=1,
    progress=gr.Progress()
):
    """
    Optimized speech generation backend running model.generate with auto-sharded FP16
    and audio tokenizer decoding on CPU. Supports Clone, Continuation, and Continuation + Clone modes.
    """
    n_vq = 32
    max_new_tokens_val = 32768

    if not text.strip():
        return None, "Error: Text input cannot be empty."
    
    status_log = []
    start_time = time.time()
    
    try:
        progress(0, desc="Preprocessing input...")
        status_log.append("🔄 Step 1: Preprocessing input...")
        
        # Determine if expected tokens is active
        duration_enabled = bool(duration_control_enabled and supports_duration_control(mode_with_reference))
        expected_tokens = int(expected_tokens_val) if duration_enabled else None
        
        lang_code = None if language == "Auto (omit)" else language
        
        status_log.append(f"  Language Tag: {language}")
        if reference_audio:
            status_log.append(f"  Voice reference: {reference_audio}")
            status_log.append(f"  Reference Mode: {mode_with_reference}")
        else:
            status_log.append("  Mode: Direct Generation (No reference audio)")
            
        if expected_tokens is not None:
            status_log.append(f"  Expected tokens (duration control): {expected_tokens}")
        status_log.append(f"  Max new tokens (budget): {max_new_tokens_val}")
        
        user_kwargs = {"text": text, "language": lang_code}
        if expected_tokens is not None:
            user_kwargs["tokens"] = int(expected_tokens)
        
        # Format the conversations list and determine generation mode
        if not reference_audio:
            message = processor.build_user_message(**user_kwargs)
            conversations = [[message]]
            mode = "generation"
        else:
            if mode_with_reference == "Clone":
                clone_kwargs = dict(user_kwargs)
                clone_kwargs["reference"] = [reference_audio]
                message = processor.build_user_message(**clone_kwargs)
                conversations = [[message]]
                mode = "generation"
            elif mode_with_reference == "Continuation":
                message = processor.build_user_message(**user_kwargs)
                assistant_message = processor.build_assistant_message(audio_codes_list=[reference_audio])
                conversations = [[message, assistant_message]]
                mode = "continuation"
            else: # Continuation + Clone
                continue_clone_kwargs = dict(user_kwargs)
                continue_clone_kwargs["reference"] = [reference_audio]
                message = processor.build_user_message(**continue_clone_kwargs)
                assistant_message = processor.build_assistant_message(audio_codes_list=[reference_audio])
                conversations = [[message, assistant_message]]
                mode = "continuation"
        
        # Tokenize and format inputs
        batch = processor(conversations, mode=mode, n_vq=32)
        input_ids = batch["input_ids"].to(device_model)
        attention_mask = batch["attention_mask"].to(device_model)
        

        
        # Fix temperature boundaries to prevent division by zero in sampling
        if audio_temp == 0.0:
            audio_temp = 0.001
        
        # Clear VRAM cache before running generation
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            gc.collect()
            
        progress(0.3, desc="Generating speech tokens (Autoregression)...")
        status_log.append(f"🧠 Step 2: Generating speech tokens on {device_model}...")
        
        with torch.no_grad():
            outputs = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=int(max_new_tokens_val),
                text_temperature=float(text_temp),
                text_top_p=float(text_top_p),
                text_top_k=int(text_top_k),
                audio_temperature=float(audio_temp),
                audio_top_p=float(audio_top_p),
                audio_top_k=int(audio_top_k),
                audio_repetition_penalty=float(audio_repetition_penalty)
            )
            
        gen_time = time.time() - start_time
        status_log.append(f"  Generated tokens in {gen_time:.2f} seconds.")
        
        progress(0.8, desc="Decoding audio tokens...")
        status_log.append(f"🔊 Step 3: Moving tokens to {device_tok} and decoding audio...")
        
        # Move output tokens list safely to GPU 0
        outputs_tok = to_device(outputs, device_tok)
        

            
        decoded_messages = processor.decode(outputs_tok)
        if not decoded_messages:
            return None, "Error: Failed to decode output tokens from the model."
            
        # Extract the stereo waveform with empty check safety
        if not decoded_messages[0].audio_codes_list:
            gen_text_ids = outputs_tok[0][1][:, 0] if (outputs_tok and len(outputs_tok) > 0) else None
            gen_text_decoded = processor.tokenizer.decode(gen_text_ids) if gen_text_ids is not None else "Unknown"
            err_msg = (
                "❌ Error: Model generated text but failed to synthesize any audio codes.\n"
                f"Generated Text Output: {gen_text_decoded}\n"
                "Please verify that prompt language matches, check hyperparameters (e.g. reduce temperatures), "
                "or try a different input text."
            )
            return None, err_msg
            
        audio = decoded_messages[0].audio_codes_list[0]
        
        # Ensure dimensions match: [channels, samples]
        if audio.ndim == 1:
            audio = audio.unsqueeze(0)
            
        # Clear tensor allocation references
        del outputs, input_ids, attention_mask, batch, decoded_messages
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            gc.collect()
            
        # Adjust playback speed if requested via resampling
        if speed != 1.0:
            progress(0.9, desc="Adjusting speech speed...")
            status_log.append(f"🏃 Speed adjustment: Resampling audio to {speed}x...")
            sample_rate = processor.model_config.sampling_rate
            new_sample_rate = int(sample_rate * speed)
            resampler = torchaudio.transforms.Resample(
                orig_freq=sample_rate,
                new_freq=new_sample_rate
            )
            audio_resampled = resampler(audio).squeeze(0)
            resampler_back = torchaudio.transforms.Resample(
                orig_freq=new_sample_rate,
                new_freq=sample_rate
            )
            audio = resampler_back(audio_resampled.unsqueeze(0))
            
        progress(0.95, desc="Saving audio file...")
        # Save generated audio to a temporary file
        temp_file = tempfile.NamedTemporaryFile(suffix=".wav", delete=False)
        output_path = temp_file.name
        temp_file.close()
        
        sampling_rate = processor.model_config.sampling_rate
        torchaudio.save(
            output_path, 
            audio.detach().cpu().to(torch.float32), 
            sampling_rate
        )
        
        duration = audio.shape[-1] / sampling_rate
        rtf = gen_time / duration if duration > 0 else 0
        
        status_log.append(f"✅ Success: Generated {duration:.1f}s of audio in {gen_time:.1f}s (RTF: {rtf:.2f}x) at {sampling_rate} Hz!")
        progress(1.0, desc="Done!")
        return output_path, "\n".join(status_log)
        
    except Exception as e:
        err_msg = f"❌ Error during generation: {str(e)}\n{traceback.format_exc()}"
        return None, err_msg

# Branded CSS
custom_css = """
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;600;700&display=swap');
* { font-family: 'Inter', sans-serif !important; }
.gradio-container { max-width: 1000px !important; margin: auto !important; }
.brand-header { text-align: center; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 28px; border-radius: 15px; margin-bottom: 20px; box-shadow: 0 10px 25px rgba(102,126,234,0.3); }
.brand-title { color: white; font-size: 2em; font-weight: 700; margin: 0 0 6px 0; }
.brand-subtitle { color: rgba(255,255,255,0.88); font-size: 1em; margin-bottom: 16px; }
.btn-row { display: flex; justify-content: center; gap: 10px; flex-wrap: wrap; }
.social-btn { display: inline-flex; align-items: center; justify-content: center; min-width: 150px; padding: 10px 18px; border-radius: 10px; font-weight: 700; font-size: 13px; text-decoration: none; color: white; white-space: nowrap; transition: transform 0.2s, box-shadow 0.2s; }
.social-btn:hover { transform: translateY(-2px); box-shadow: 0 6px 16px rgba(0,0,0,0.3); }
.yt-btn  { background: #FF0000; box-shadow: 0 4px 12px rgba(255,0,0,0.3); }
.x-btn   { background: #000000; box-shadow: 0 4px 12px rgba(0,0,0,0.25); }
.sup-btn { background: linear-gradient(135deg,#f6d365,#fda085); box-shadow: 0 4px 12px rgba(253,160,133,0.35); }
button.primary { background: linear-gradient(135deg, #667eea 0%, #764ba2 100%) !important; color: white !important; font-weight: 600 !important; border-radius: 12px !important; }
#stop-btn { background: linear-gradient(135deg, #ef4444 0%, #b91c1c 100%) !important; color: white !important; font-weight: 600 !important; border-radius: 12px !important; }
#clear-btn { background: linear-gradient(135deg, #6b7280 0%, #374151 100%) !important; color: white !important; font-weight: 600 !important; border-radius: 12px !important; }
.footer { text-align: center; padding: 20px; margin-top: 30px; border-top: 2px solid #e5e7eb; color: #6b7280; }
"""

with gr.Blocks(title="MOSS-TTS v1.5 - AIQUEST Academy", theme=gr.themes.Soft(), css=custom_css) as demo:
    # Branded Header Component
    gr.HTML("""
    <div class="brand-header">
      <div class="brand-title">🎤 MOSS-TTS v1.5 - Foundation Model</div>
      <div class="brand-subtitle">Created by <strong>AIQUEST Academy</strong> &nbsp;|&nbsp; Kaggle T4 x2 GPU Edition · Sharded FP16 Precision</div>
      <div class="btn-row">
        <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1" target="_blank" class="social-btn yt-btn">▶ Subscribe on YouTube</a>
        <a href="https://x.com/aiquestacademy" target="_blank" class="social-btn x-btn">𝕏 Follow on X</a>
        <a href="https://aiquest.site" target="_blank" class="social-btn sup-btn">❤️ Support My Work</a>
      </div>
    </div>
    """)

    gr.HTML("<p style='text-align: center; margin-top: 10px;'>High-Fidelity 48 kHz stereo Speech Synthesis and zero-shot Voice Cloning (Sharded FP16 Precision)</p>")

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 📥 Input Configuration")
            text_input = gr.Textbox(
                label="Synthesize Text",
                placeholder="Type the text you want to speak. Support inline pauses using e.g., [pause 1.5s] syntax. In continuation modes, prepend reference audio transcript.",
                lines=4,
                value="This is MOSS text-to-speech, running on Kaggle dual T4 GPU free tier. Please subscribe to AIQUEST for more free stuff!"
            )
            lang_input = gr.Dropdown(
                choices=LANGUAGES,
                label="Language Tag (Optional)",
                value="Auto (omit)"
            )
            ref_audio_input = gr.Audio(
                label="Reference Audio (Optional, for zero-shot Voice Cloning or Continuation)",
                type="filepath"
            )
            mode_with_reference = gr.Radio(
                choices=["Clone", "Continuation", "Continuation + Clone"],
                value="Clone",
                label="Mode with Reference Audio",
                info="If no reference audio is uploaded, Direct Generation will be used automatically. Note: Continuation modes require reference transcript at start of text."
            )
            mode_hint = gr.Markdown("Current mode: **Direct Generation** (no reference audio uploaded)")
            
            with gr.Accordion("⚙️ Advanced Settings", open=False):
                with gr.Row():
                    speed_slider = gr.Slider(
                        minimum=0.5, maximum=2.0, value=1.0, step=0.1,
                        label="Speed Factor"
                    )
                
                duration_control_enabled = gr.Checkbox(
                    value=False,
                    label="Enable Duration Control (Expected Audio Tokens)"
                )
                expected_tokens = gr.Slider(
                    minimum=1,
                    maximum=1,
                    step=1,
                    value=1,
                    label="expected_tokens",
                    visible=False
                )
                duration_hint = gr.Markdown("Duration control is disabled.")

                with gr.Row():
                    text_temp_slider = gr.Slider(minimum=0.1, maximum=2.0, value=1.5, step=0.1, label="text_temp")
                    text_top_p_slider = gr.Slider(minimum=0.1, maximum=1.0, value=1.0, step=0.05, label="text_top_p")
                    text_top_k_slider = gr.Slider(minimum=1, maximum=100, value=50, step=1, label="text_top_k")
                with gr.Row():
                    audio_temp_slider = gr.Slider(minimum=0.1, maximum=3.0, value=1.7, step=0.05, label="audio_temp")
                    audio_top_p_slider = gr.Slider(minimum=0.1, maximum=1.0, value=0.9, step=0.01, label="audio_top_p")
                with gr.Row():
                    audio_top_k_slider = gr.Slider(minimum=1, maximum=200, value=25, step=1, label="audio_top_k")
                    audio_rep_pen_slider = gr.Slider(minimum=0.8, maximum=2.0, value=1.0, step=0.05, label="audio_repetition_penalty")

            # Gradio Buttons (Always 3: Generate, Stop, Clear)
            with gr.Row():
                gen_btn = gr.Button("🔊 Generate Audio", variant="primary", size="lg", elem_id="gen-btn")
                stop_btn = gr.Button("🛑 Stop", variant="secondary", size="lg", elem_id="stop-btn")
                clear_btn = gr.Button("🗑️ Clear", variant="secondary", size="lg", elem_id="clear-btn")

        with gr.Column(scale=1):
            gr.Markdown("### 📤 Output Results")
            audio_output = gr.Audio(
                label="Generated Speech",
                interactive=False
            )
            log_output = gr.Textbox(
                label="Process Log",
                lines=14,
                interactive=False
            )

    # Branded Footer
    gr.HTML(
        "<div class='footer'>"
        "<p style='margin:0; text-align:center'>© 2026 AIQUEST Academy. Powered by OpenMOSS-Team MOSS-TTS v1.5.</p>"
        "</div>"
    )

    # Wire up reference audio and mode hint renders
    ref_audio_input.change(
        fn=render_mode_hint,
        inputs=[ref_audio_input, mode_with_reference],
        outputs=[mode_hint]
    )
    mode_with_reference.change(
        fn=render_mode_hint,
        inputs=[ref_audio_input, mode_with_reference],
        outputs=[mode_hint]
    )

    # Wire up duration control dynamic calculations
    duration_control_enabled.change(
        fn=update_duration_controls,
        inputs=[duration_control_enabled, text_input, expected_tokens, mode_with_reference],
        outputs=[expected_tokens, duration_hint, duration_control_enabled],
    )
    text_input.change(
        fn=update_duration_controls,
        inputs=[duration_control_enabled, text_input, expected_tokens, mode_with_reference],
        outputs=[expected_tokens, duration_hint, duration_control_enabled],
    )
    mode_with_reference.change(
        fn=update_duration_controls,
        inputs=[duration_control_enabled, text_input, expected_tokens, mode_with_reference],
        outputs=[expected_tokens, duration_hint, duration_control_enabled],
    )

    # Wire up button event listeners
    gen_event = gen_btn.click(
        fn=generate_speech,
        inputs=[
            text_input, lang_input, ref_audio_input, mode_with_reference, speed_slider,
            text_temp_slider, text_top_p_slider, text_top_k_slider,
            audio_temp_slider, audio_top_p_slider, audio_top_k_slider, audio_rep_pen_slider,
            duration_control_enabled, expected_tokens
        ],
        outputs=[audio_output, log_output]
    )
    
    stop_btn.click(
        fn=None,
        cancels=[gen_event]
    )
    
    clear_btn.click(
        fn=lambda: (None, "", "This is MOSS text-to-speech, running on Kaggle dual T4 GPU free tier. Please subscribe to AIQUEST for more free stuff!", "Auto (omit)", None, "Clone", 1.0, 1.2, 1.0, 50, 1.7, 0.9, 25, 1.0, False, 1),
        inputs=[],
        outputs=[
            audio_output, log_output, text_input, lang_input, ref_audio_input, mode_with_reference, speed_slider,
            text_temp_slider, text_top_p_slider, text_top_k_slider,
            audio_temp_slider, audio_top_p_slider, audio_top_k_slider, audio_rep_pen_slider,
            duration_control_enabled, expected_tokens
        ]
    )

# Launch Gradio interface with public sharing enabled
demo.queue(max_size=3)
demo.launch(share=True, show_error=True)